In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Nat_Gas.csv')

In [3]:
df.head()

,Dates,Prices
0,10/31/20,10.1
1,11/30/20,10.3
2,12/31/20,11.0
3,1/31/21,10.9
4,2/28/21,10.9


In [6]:
df['Dates'] = pd.to_datetime(df['Dates'])

In [7]:
df =df.sort_values('Dates').reset_index(drop = True)

In [8]:
df.head()

,Dates,Prices
0,2020-10-31,10.1
1,2020-11-30,10.3
2,2020-12-31,11.0
3,2021-01-31,10.9
4,2021-02-28,10.9


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Dates   48 non-null     datetime64[ns]
 1   Prices  48 non-null     float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 900.0 bytes


In [10]:
import pandas as pd


def contract_price_estimation(
    injection_schedule,
    withdrawal_schedule,
    injection_rate,
    withdrawal_rate,
    max_capacity,
    storage_cost_per_month,
    inj_with_cost_per_unit,
    price_estimator,
):
    # Convert to dataframe and label types
    inj_df = pd.DataFrame(injection_schedule, columns=["Date", "Volume"])
    inj_df["Type"] = "inj"

    with_df = pd.DataFrame(withdrawal_schedule, columns=["Date", "Volume"])
    with_df["Type"] = "with"

    events = (
        pd.concat([inj_df, with_df]).sort_values("Date").reset_index(drop=True)
    )
    events["Date"] = pd.to_datetime(events["Date"])

    current_inventory = 0
    total_injected = 0
    total_withdrawn = 0
    purchase_cost = 0
    sales_revenue = 0

    # Process each injection and withdrawal event
    for idx, row in events.iterrows():
        date_str = row["Date"].strftime("%Y-%m-%d")
        vol = row["Volume"]
        price = price_estimator(date_str)

        if row["Type"] == "inj":
            if vol > injection_rate:
                raise ValueError(
                    f"Injection rate exceeded on {date_str}: {vol} > {injection_rate}"
                )
            current_inventory += vol
            if current_inventory > max_capacity:
                raise ValueError(
                    f"Max storage capacity exceeded on {date_str}: {current_inventory} > {max_capacity}"
                )

            purchase_cost += vol * price
            total_injected += vol

        elif row["Type"] == "with":
            if vol > withdrawal_rate:
                raise ValueError(
                    f"Withdrawal rate exceeded on {date_str}: {vol} > {withdrawal_rate}"
                )
            current_inventory -= vol
            if current_inventory < 0:
                raise ValueError(
                    f"Negative inventory on {date_str}! Cannot withdraw more than stored."
                )

            sales_revenue += vol * price
            total_withdrawn += vol

    # Calculate fixed storage fees based on total duration (inclusive of months)
    start_date = events["Date"].min()
    end_date = events["Date"].max()
    duration_months = ((end_date.year - start_date.year) * 12 + (end_date.month - start_date.month) + 1)
    total_storage_cost = duration_months * storage_cost_per_month

    # Total variable fee
    total_variable_fee = (total_injected + total_withdrawn) * inj_with_cost_per_unit

    # Net contract value
    contract_value = (sales_revenue - purchase_cost - total_storage_cost - total_variable_fee)
    return {
        "contract_value": round(contract_value, 2),
        "sales_revenue": round(sales_revenue, 2),
        "purchase_cost": round(purchase_cost, 2),
        "storage_cost": round(total_storage_cost, 2),
        "variable_fee": round(total_variable_fee, 2),
        "ending_inventory": current_inventory,
    }

In [16]:
# Setup Mock Price Estimator
def price_lookup(date_str):
    prices = {
        "2026-04-15": 2.50,  # Spring buy
        "2026-05-15": 2.80,  # Spring buy
        "2026-11-15": 5.50,  # Winter sell
    }
    return prices[date_str]


# Sample Inputs
inj_schedule = [("2026-04-15", 500), ("2026-05-15", 500)]
with_schedule = [("2026-11-15", 800)]

result = contract_price_estimation(
    injection_schedule=inj_schedule,
    withdrawal_schedule=with_schedule,
    injection_rate=600,
    withdrawal_rate=1000,
    max_capacity=1500,
    storage_cost_per_month=100,
    inj_with_cost_per_unit=0.10,
    price_estimator=price_lookup,
)

print(result)

{'contract_value': 770.0, 'sales_revenue': 4400.0, 'purchase_cost': 2650.0, 'storage_cost': 800, 'variable_fee': 180.0, 'ending_inventory': 200}
